# CAUTI Prediction with PyHealth: End-to-End Tutorial

**Catheter-Associated Urinary Tract Infection (CAUTI)** is one of the most common healthcare-acquired
infections, accounting for ~30% of all hospital-acquired infections. Early prediction of CAUTI
risk enables proactive clinical intervention and better patient outcomes.

This tutorial walks through the complete PyHealth pipeline for CAUTI prediction on MIMIC-IV:

| Step | What we do |
|------|------------|
| 1. Data Loading | Load MIMIC-IV EHR tables with `MIMIC4Dataset` |
| 2. Task Setup | Apply `CatheterAssociatedInfectionPredictionMIMIC4` to generate labeled samples |
| 3. Splitting | Patient-level train/val/test split |
| 4. Model | Instantiate and train a `Transformer` model |
| 5. Evaluation | Compute PR-AUC, ROC-AUC, Precision, F1, Accuracy on held-out test set |
| 6. Interpretability | Use Chefer Relevance to identify which feature groups drove the prediction |

---

### Task Design Notes

The task class encodes several clinical design decisions:

- **Positive label**: An admission is CAUTI-positive if it contains an *unconditional* catheter
  infection code (e.g., ICD-10 T83.511A) OR a *conditional* infection code (e.g., N39.0 - UTI)
  co-occurring with any catheter procedure/diagnosis code.

- **Temporal leakage prevention** (`same_visit=True`): For the infection admission, labs after
  admission time are excluded (set to a zero vector). This prevents the model from seeing lab
  abnormalities that only appeared *because of* the infection.

- **Per-admission negative sampling**: Every non-CAUTI admission from a catheter-exposed patient
  generates a separate negative training sample (instead of one negative per patient).

- **Feature encoding**: Diagnoses and procedures are mapped to CCS/CCSR groupings; drugs are
  mapped to ATC-3 codes; labs are averaged into a 10-dimensional vector per admission.

## Imports

In [ ]:
import torch
import matplotlib.pyplot as plt
from collections import Counter

from pyhealth.datasets import MIMIC4Dataset, get_dataloader, split_by_patient
from pyhealth.tasks import CatheterAssociatedInfectionPredictionMIMIC4
from pyhealth.models import Transformer
from pyhealth.trainer import Trainer
from pyhealth.interpret.methods.chefer import CheferRelevance

---
## Step 1: Load MIMIC-IV EHR Data

`MIMIC4Dataset` reads the raw CSV files, applies joins (e.g., attaching `dischtime` to ICD
codes), and builds a per-patient `Patient` object with a unified event timeline.

We load five tables:
- **admissions** - hospital admission metadata (admit/discharge times, demographics)
- **diagnoses_icd** - ICD-9/10 diagnosis codes per admission
- **procedures_icd** - ICD-9/10 procedure codes per admission
- **prescriptions** - drug orders with NDC codes and start/stop times
- **labevents** - laboratory results with per-draw chart timestamps

On subsequent runs, the dataset is loaded from `cache_dir` without re-scanning the CSVs.

In [ ]:
MIMIC4_ROOT = "/shared/rsaas/physionet.org/files/mimiciv/2.2"
CACHE_DIR   = "/shared/eng/pyhealth_agent/cache"

if __name__ == '__main__':
    base_dataset = MIMIC4Dataset(
        ehr_root=MIMIC4_ROOT,
        cache_dir=CACHE_DIR,
        ehr_tables=[
            "admissions",
            "diagnoses_icd",
            "procedures_icd",
            "prescriptions",
            "labevents",
        ],
        num_workers=4,
    )

---
## Step 2: Apply the CAUTI Prediction Task

`CatheterAssociatedInfectionPredictionMIMIC4` iterates over each patient's admission history
and produces **samples** - one dict per predictable event.

Each sample contains:
- `conditions` - nested list of CCS diagnosis tokens, one sublist per prior admission
- `procedures` - nested list of CCSPROC procedure tokens, one sublist per prior admission
- `drugs` - nested list of ATC-3 drug tokens, one sublist per prior admission
- `labs` - nested list of 10-dim lab vectors (Sodium, Potassium, Chloride, Bicarbonate,
  Glucose, Calcium, Magnesium, Anion Gap, Osmolality, Phosphate), one per prior admission
- `label` - 1 (CAUTI on target admission) or 0 (no CAUTI)

Key task parameters:
- `same_visit=True` - include the infection admission's non-infection codes in features
  (with labs zeroed out to prevent leakage)
- `map_ccscm=True` - map raw ICD codes to CCS/CCSR groupings (reduces vocabulary size)

### Task Class Architecture

Task classes in PyHealth are the bridge between raw EHR data and model-ready samples.
The full CAUTI task hierarchy lives in
[`pyhealth/tasks/catheter_infection.py`](../../../pyhealth/tasks/catheter_infection.py):

```
BaseTask (abstract)
└── _CatheterInfectionBase          ← shared code set definitions + helper methods
    ├── CatheterAssociatedInfectionPredictionStageNetMIMIC4
    ├── CatheterAssociatedInfectionPredictionMIMIC4          ← used in this tutorial
    ├── CatheterAssociatedInfectionPredictionStageNetMIMIC4DualContext
    └── CatheterAssociatedInfectionPredictionMIMIC4DualContext
```

Every task class exposes three key pieces of contract:

**`input_schema`** — maps each feature field name to its PyHealth data type.
The schema drives two things: (1) how `SampleDataset` collates samples into batches,
and (2) which embedding/processing pipeline the model uses for each field.

| Type string | Meaning |
|---|---|
| `"nested_sequence"` | `List[List[str]]` — one token list per admission, variable length |
| `"nested_sequence_floats"` | `List[List[float]]` — one float vector per admission |
| `"stagenet"` | `(List[float], List[List[str]])` — (time deltas, per-step token lists) |
| `"stagenet_tensor"` | `(List[float], List[List[float]])` — (time deltas, per-step float vectors) |
| `"binary"` | scalar 0/1 label |

**`output_schema`** — maps each label field to its label type (`"binary"`, `"multiclass"`, etc.).
This tells the trainer which loss function and which metrics apply.

**`__call__(patient) -> List[Dict]`** — the core inclusion logic.
It receives one `Patient` object and returns zero or more sample dicts.
Empty list = this patient contributes nothing (e.g., no admissions, no catheter evidence).

The inclusion criteria encoded in `__call__`:
1. Patient must have at least one `admissions` event.
2. Each admission is classified as *infection* or *non-infection* using `_determine_positive_label()`.
3. **Positive samples**: generated for infection admissions only. `same_visit=True` appends
   the infection admission's features (with infection codes masked) to the feature window.
   Suffix augmentation then produces additional positive samples by progressively truncating
   earlier admissions — turning one infection event into `n_prior_admissions + 1` samples.
4. **Negative samples**: generated only for *catheter-evidence patients* (those who had at
   least one catheter code anywhere in their history). Each non-CAUTI admission after the
   first catheter code emits one negative sample — per admission, not per patient.

---

**How `set_task(task, num_workers=N)` parallelizes `__call__`:**

```
set_task(task, num_workers=4)
│
├── Collects all patient IDs from the global event DataFrame
├── Splits IDs into N equal-ish chunks (itertools.batched)
│
├── Spawns N worker processes (multiprocessing "spawn" context)
│   Each worker:
│   ├── Receives its chunk of patient IDs + a LazyFrame of all events
│   ├── For each patient ID:
│   │   ├── Reconstructs Patient object from LazyFrame slice
│   │   └── Calls task(patient)  ← your __call__ runs here
│   └── Writes resulting sample dicts to a LitData binary shard
│
└── Merges shards → SampleDataset (memory-mapped, zero-copy reads)
```

> **Notebook caveat**: PyHealth auto-detects Jupyter and forces `num_workers=1`
> (sequential) because the `spawn` start method requires a `__main__` guard that
> interactive notebooks cannot provide. The `if __name__ == '__main__':` guards
> in the cells below are present so this code can also be run as a plain script.

The DualContext variant below shows how a task class can compose two `same_visit` modes
into a single callable — doubling the training signal with no extra EHR scanning.

In [ ]:
# ============================================================
# Annotated inline copy of CatheterAssociatedInfectionPredictionMIMIC4DualContext
#
# Source: pyhealth/tasks/catheter_infection.py
# This cell is meant to be READ, not necessarily re-run.
# The class is already imported from pyhealth.tasks above.
# ============================================================

from typing import Any, Dict, List
from pyhealth.tasks.catheter_infection import (
    _CatheterInfectionBase,
    CatheterAssociatedInfectionPredictionMIMIC4,
)


class CatheterAssociatedInfectionPredictionMIMIC4DualContext(_CatheterInfectionBase):
    """Dual-context nested-sequence CAUTI task.

    Emits BOTH same_visit modes for every patient in a single pass.
    This doubles the training signal without scanning EHR events twice:
      - visit_mode="current": same-visit features (infection codes masked, labs zeroed)
      - visit_mode="next":    prior-admissions-only features (strictest leakage prevention)
    """

    task_name: str = "CatheterAssociatedInfectionPredictionMIMIC4DualContext"

    # ------------------------------------------------------------------
    # input_schema
    # ------------------------------------------------------------------
    # Declares the feature fields and their data types.
    # PyHealth uses this to:
    #   1. Build the right embedding layer per field in the model.
    #   2. Collate variable-length lists into padded tensors in get_dataloader().
    #
    # "nested_sequence"       → List[List[str]]   — token sequences per admission
    # "nested_sequence_floats"→ List[List[float]] — float vectors per admission
    # ------------------------------------------------------------------
    input_schema: Dict[str, str] = {
        "conditions": "nested_sequence",    # CCS-CM diagnosis tokens, one list per visit
        "procedures": "nested_sequence",    # CCS-PCS procedure tokens, one list per visit
        "drugs":      "nested_sequence",    # ATC Level-3 drug tokens, one list per visit
        "labs":       "nested_sequence_floats",  # 10-dim mean lab vectors, one per visit
    }

    # ------------------------------------------------------------------
    # output_schema
    # ------------------------------------------------------------------
    # Declares the label field and its type.
    # "binary" → BCE loss, PR-AUC / ROC-AUC metrics in Trainer.
    # ------------------------------------------------------------------
    output_schema: Dict[str, str] = {"label": "binary"}

    def __init__(self, map_ccscm: bool = True):
        """
        Args:
            map_ccscm: Map raw ICD codes to CCS/CCSR groupings (True by default).
                       Reduces vocabulary size from ~70k ICD codes to ~300 CCS groups,
                       which substantially improves embedding quality on small cohorts.
        """
        self.map_ccscm = map_ccscm
        # task_name is used as a cache key — changing any parameter should change the name.
        self.task_name = self._task_name_with_map_ccscm(
            type(self).task_name,
            self.map_ccscm,
        )

    @staticmethod
    def _tag_samples(
        samples: List[Dict[str, Any]], visit_mode: str
    ) -> List[Dict[str, Any]]:
        """Append a visit_mode tag and a mode suffix to each sample's record_id.

        This is what lets a single SampleDataset carry both modes without
        record_id collisions.  The model can then filter by visit_mode at
        evaluation time to compare same-visit vs. next-visit performance.
        """
        suffix = "current" if visit_mode == "current" else "next"
        tagged = []
        for sample in samples:
            updated = dict(sample)
            updated["record_id"] = f"{sample['record_id']}_{suffix}"
            updated["visit_mode"] = visit_mode   # extra field — ignored by model, kept for analysis
            tagged.append(updated)
        return tagged

    def __call__(self, patient: Any) -> List[Dict[str, Any]]:
        """Inclusion logic: called once per patient by each set_task() worker.

        Inclusion criteria (both modes share the same patient gate):
          - Patient must have ≥1 admissions event.
          - Positive samples: admissions containing CAUTI infection codes
            (unconditional tier, or conditional tier + catheter co-occurrence).
          - Negative samples: non-infection admissions from catheter-evidence patients,
            one sample per admission (not one per patient).

        The DualContext __call__ delegates the heavy lifting to the base task class
        (run twice with same_visit=True and same_visit=False), then merges and tags.
        No EHR events are re-fetched — each base task call reconstructs features
        independently from the same Patient object in memory.
        """
        # ------------------------------------------------------------------
        # Run both modes.  Each instantiates its own CatheterAssociatedInfection-
        # PredictionMIMIC4 with the appropriate same_visit flag.  The base task's
        # __call__ does all the admission-loop logic, lab vector building, masking,
        # suffix augmentation, and negative sampling.
        # ------------------------------------------------------------------
        current_task = CatheterAssociatedInfectionPredictionMIMIC4(
            same_visit=True,   # include infection-admission features (infection codes masked)
            map_ccscm=self.map_ccscm,
        )
        next_task = CatheterAssociatedInfectionPredictionMIMIC4(
            same_visit=False,  # prior admissions only — strictest leakage prevention
            map_ccscm=self.map_ccscm,
        )

        # Each returns List[Dict] for this one patient.
        current_samples = self._tag_samples(current_task(patient), "current")
        next_samples    = self._tag_samples(next_task(patient),    "next")

        # The concatenated list is what set_task() writes to the LitData shard.
        # A patient with 3 prior admissions and 1 CAUTI event produces:
        #   current: 1 positive (base) + 3 augmented + N negatives
        #   next:    same structure, features shifted one admission earlier
        return current_samples + next_samples


# Spot-check: verify the schema is correct
dc_task = CatheterAssociatedInfectionPredictionMIMIC4DualContext(map_ccscm=True)
print("task_name  :", dc_task.task_name)
print("input_schema :", dc_task.input_schema)
print("output_schema:", dc_task.output_schema)

In [ ]:
task = CatheterAssociatedInfectionPredictionMIMIC4(same_visit=True, map_ccscm=True)

if __name__ == '__main__':
    sample_dataset = base_dataset.set_task(task, num_workers=4)

    print(f'Total samples: {len(sample_dataset)}')

    labels = [s['label'] for s in sample_dataset.samples]
    label_counts = Counter(labels)
    print(f'Positive (CAUTI): {label_counts[1]}')
    print(f'Negative:         {label_counts[0]}')
    if label_counts[1] > 0:
        print(f'Imbalance ratio:  {label_counts[0] / label_counts[1]:.1f}:1')

In [ ]:
# Inspect a single sample
sample = sample_dataset.samples[0]
print('Sample keys:', list(sample.keys()))
print('\nNumber of prior admissions in this sample:', len(sample['conditions']))
print('Conditions (first visit):', sample['conditions'][0][:5], '...')
print('Labs (first visit, 10-dim):', [round(x, 3) for x in sample['labs'][0]])
print('Label:', sample['label'])

---
## Step 3: Patient-Level Train / Val / Test Split

`split_by_patient` ensures that all samples from a given patient appear in exactly one split.
This is critical for clinical prediction tasks - patient-level separation prevents the model
from learning patient-specific patterns rather than generalizable clinical signals.

In [ ]:
train_dataset, val_dataset, test_dataset = split_by_patient(
    sample_dataset, [0.8, 0.1, 0.1], seed=42
)
print(f'Train: {len(train_dataset)} samples')
print(f'Val:   {len(val_dataset)} samples')
print(f'Test:  {len(test_dataset)} samples')

train_loader = get_dataloader(train_dataset, batch_size=8, shuffle=True)
val_loader   = get_dataloader(val_dataset,   batch_size=8, shuffle=False)
test_loader  = get_dataloader(test_dataset,  batch_size=8, shuffle=False)

---
## Step 4: Build and Train the Transformer Model

PyHealth's `Transformer` applies multi-head self-attention independently to each feature
sequence (conditions, procedures, drugs, labs), then concatenates the [CLS] token embeddings
and passes them through a linear classification head.

The model infers its input schema directly from `sample_dataset`, so there is no need to
manually specify `feature_keys` or `label_key`.

**Hyperparameters:**
- `embedding_dim=128` - token embedding size
- `heads=4` - attention heads per layer
- `num_layers=2` - stacked transformer blocks
- `dropout=0.3` - regularization
- `max_seq_len=1024` - maximum sequence length (positional encoding)

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model = Transformer(
    dataset=sample_dataset,
    embedding_dim=128,
    heads=4,
    num_layers=2,
    dropout=0.3,
    max_seq_len=1024,
)
print(model)

In [ ]:
trainer = Trainer(
    model=model,
    device=device,
    metrics=['pr_auc', 'roc_auc', 'precision', 'f1', 'accuracy'],
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=20,
    monitor='pr_auc',
    monitor_criterion='max',
    optimizer_params={'lr': 1e-4},
    patience=5,
)

---
## Step 5: Evaluate on the Test Set

After training, `trainer.evaluate()` runs inference on the held-out test set and returns
all tracked metrics.

Given the class imbalance (~10-30:1 negatives to positives), **PR-AUC** is the primary
metric of interest - it measures precision-recall trade-off and is more informative than
ROC-AUC for rare positive events.

In [ ]:
results = trainer.evaluate(test_loader)

print('Test Set Results:')
print('-' * 30)
for metric, value in results.items():
    print(f'  {metric:<12}: {value:.4f}')

---
## Step 6: Interpretability with Chefer Relevance

**Chefer Relevance** (Chefer et al., 2021) propagates attention weights weighted by their
gradients back through the transformer layers to produce a relevance score per input token.
Here we aggregate token-level scores to get a **feature-group relevance** - one scalar per
input feature key (conditions, procedures, drugs, labs).

A higher relevance score means that feature group had more influence on the prediction for
this particular sample.

> **Note:** Relevance scores are sample-specific. The chart below reflects one patient's
> prediction, not population-level feature importance.

In [ ]:
model.eval()

# Use batch_size=1 for single-sample interpretability
single_loader = get_dataloader(test_dataset, batch_size=1, shuffle=False)
data_iter = iter(single_loader)
sample_batch = next(data_iter)

# CheferRelevance expects class_index to specify the target class
sample_batch['class_index'] = sample_batch['label']

relevance = CheferRelevance(model)
rel_scores = relevance.get_relevance_matrix(**sample_batch)

true_label = int(sample_batch['label'][0])
print(f"True label: {'CAUTI (positive)' if true_label == 1 else 'No CAUTI (negative)'}")
print('\nFeature group relevance scores:')
for k in sorted(rel_scores.keys()):
    score = float(rel_scores[k].view(-1).detach().cpu())
    print(f'  {k:<15}: {score:.4f}')

In [ ]:
feature_colors = {
    'conditions': '#4C72B0',
    'procedures': '#55A868',
    'drugs':      '#C44E52',
    'labs':       '#8172B2',
}

keys   = sorted(rel_scores.keys())
values = [float(rel_scores[k].view(-1).detach().cpu()) for k in keys]
colors = [feature_colors.get(k, '#999999') for k in keys]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(keys, values, color=colors, edgecolor='white', linewidth=0.8)
ax.set_xlabel('Feature Group', fontsize=12)
ax.set_ylabel('Chefer Relevance Score', fontsize=12)
label_str = 'CAUTI' if true_label == 1 else 'No CAUTI'
ax.set_title(f'Feature Relevance - True label: {label_str}', fontsize=13)
ax.bar_label(bars, fmt='%.3f', padding=4, fontsize=10)
ax.set_ylim(0, max(values) * 1.2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Summary

This tutorial demonstrated the complete PyHealth pipeline for CAUTI prediction:

1. **`MIMIC4Dataset`** - loads and caches structured EHR tables from MIMIC-IV
2. **`CatheterAssociatedInfectionPredictionMIMIC4`** - generates temporally sound, per-admission
   labeled samples with leakage prevention and per-admission negative sampling
3. **`split_by_patient`** - ensures no patient leakage between train/val/test
4. **`Transformer`** - multi-head attention over multi-modal EHR feature sequences
5. **`Trainer`** - standard training loop with early stopping and best-model checkpointing
6. **`CheferRelevance`** - attention-gradient interpretability showing which feature groups
   drove each prediction

### Next Steps

- Try other models in the same directory (`adacare.py`, `concare.py`, `stagenet.py`, `retain.py`)
  to compare performance
- Use `CatheterAssociatedInfectionPredictionMIMIC4DualContext` for training on both
  same-visit and next-visit prediction jointly
- Explore `examples/interpretability/shap_stagenet_mimic4.ipynb` for SHAP-based
  interpretability on the StageNet variant